# Road, Environmental and Spatial Predictors of Severe Road Collisions in London

## Preparation

- Github link: <https://github.com/chestnutwei/Coursework>
- Number of words: approximately 1,450, excluding code, code comments, tables and references
- Runtime: approximately 5–8 minutes on a standard laptop
- Coding environment: Jupyter Notebook / Anaconda
- License: Creative Commons Attribution license
- Additional libraries:
  - `openpyxl`: reading the Excel data guide
  - `scikit-learn`: classification models and evaluation
  - `matplotlib` / `seaborn`: visualisation

## Table of contents

1. [Introduction](#Introduction)
2. [Research questions](#Research-questions)
3. [Data](#Data)
4. [Methodology](#Methodology)
5. [Results and discussion](#Results-and-discussion)
6. [Conclusion](#Conclusion)
7. [References](#References)

## Introduction

Road danger remains a major transport and public health issue in London. The city's Vision Zero programme aims to eliminate deaths and serious injuries from the transport network by 2041 (Transport for London, n.d.). This project therefore focuses on severe road collisions, defined as fatal or serious injury collisions, rather than total collision counts.

Counts are strongly shaped by exposure: areas with more traffic, road length or activity tend to record more collisions. This project asks a narrower question: conditional on a police-reported injury collision having occurred, which recorded road, environmental, temporal and spatial characteristics are associated with a severe outcome?

Crash-injury severity has a long methodological literature in road safety research. Savolainen et al. (2011) survey the statistical approaches used to analyse severity outcomes, while Mannering and Bhat (2014) argue that crash analysis needs careful interpretation because unobserved factors and modelling choices strongly affect results. Edwards (1998) examined recorded weather and accident severity in England and Wales. The present project draws on this work and uses 2024 Department for Transport collision data to test whether an interpretable classification workflow can identify conditions linked to severe outcomes in London.

## Research questions

**Main research question**

How do road, environmental, temporal, and spatial characteristics influence the likelihood of severe road collisions in London?

**Sub-questions**

1. How does the proportion of severe collisions vary across London boroughs?
2. Which features contribute most to distinguishing severe from slight collisions?


## Data

The main dataset is the Department for Transport Road Safety Open Data collision file for 2024, which records personal injury collisions in Great Britain, including location, severity, road context and environmental conditions (Department for Transport, 2025a). London records are selected using `police_force` codes 1 and 48, representing the Metropolitan Police and City of London.

The original file contains 100,927 collisions and 44 variables. Filtering to London gives 21,000 collisions. The target is derived from `collision_severity`: fatal and serious collisions are coded as severe, while slight collisions are coded as non-severe. Severe collisions account for 17.2% of London records. For borough-level analysis and modelling, 33 Heathrow-coded records outside the 33 London boroughs are excluded.

Most predictors are numeric codes, so the official data guide is used to decode labels. Sentinel values -1 ("Data missing or out of range") and 99 ("unknown") are consolidated into `Missing/Unknown` before one-hot encoding. STATS19 also covers only police-reported personal injury collisions on public roads, excluding damage-only and unreported collisions (Department for Transport, 2025b).

One important caveat is that the dataset contains no exposure measure. There is no traffic volume, pedestrian or cycle flow, road length or vehicle-kilometres travelled by borough or road type. Borough-level severe-collision shares therefore cannot be read as complete measures of road danger: a borough may show a high severe share because of road design and traffic conditions, but also because of who travels there, how much traffic it carries, or differences in reporting. The analysis works around this by asking about severity conditional on a reported collision having occurred.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

collision_url = "https://raw.githubusercontent.com/chestnutwei/Coursework/main/data/dft-road-casualty-statistics-collision-2024.csv"
guide_url = "https://raw.githubusercontent.com/chestnutwei/Coursework/main/data/dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2024.xlsx"

collisions = pd.read_csv(collision_url, low_memory=False)
guide = pd.read_excel(guide_url, sheet_name="2024_code_list", engine="openpyxl")

print(f"Collision data: {collisions.shape[0]:,} rows, {collisions.shape[1]} columns")
print(f"Data guide: {guide.shape[0]:,} rows, {guide.shape[1]} columns")

### Data filtering and target construction

London collisions are selected using police force codes. The binary target `severe` equals 1 for fatal or serious collisions, and 0 for slight collisions.


In [ ]:
# London = Metropolitan Police + City of London
london = collisions[collisions["police_force"].isin([1, 48])].copy()

# severe = 1 if fatal or serious, else 0
london["severe"] = london["collision_severity"].isin([1, 2]).astype(int)

# parse date and extract month / hour for temporal patterns
london["date"] = pd.to_datetime(london["date"], dayfirst=True, errors="coerce")
london["month"] = london["date"].dt.month
london["hour"] = london["time"].astype(str).str.split(":").str[0].astype(int)

severity_summary = (
    london["severe"]
    .value_counts()
    .rename(index={0: "Slight", 1: "Fatal/Serious"})
    .to_frame("count")
)
severity_summary["share"] = (severity_summary["count"] / len(london)).round(3)

print(f"London subset: {london.shape[0]:,} collisions")
display(severity_summary)

### Decoding categorical fields

Many road and environmental fields are stored as numeric codes. They are decoded with the official data guide for clearer figures, while the modelling pipeline still treats them as categorical features.


In [ ]:
def build_code_map(field_name, table_name="collision"):
    # look up the code -> label mapping for one field from the official data guide
    sub = guide[
        (guide["table"] == table_name) &
        (guide["field name"] == field_name)
    ].dropna(subset=["code/format", "label"])

    mapping = {}
    for code_value, label in zip(sub["code/format"], sub["label"]):
        try:
            mapping[int(float(code_value))] = label
        except (ValueError, TypeError):
            continue
    return mapping


decode_cols = [
    "collision_severity",
    "road_type",
    "junction_detail",
    "junction_control",
    "pedestrian_crossing",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "special_conditions_at_site",
    "carriageway_hazards",
    "urban_or_rural_area",
    "trunk_road_flag"
]

for col in decode_cols:
    london[f"{col}_label"] = london[col].map(build_code_map(col))

# code 0 is left blank in the guide for these two; label it explicitly so it isn't dropped as NaN
london.loc[london["special_conditions_at_site"].eq(0), "special_conditions_at_site_label"] = "None"
london.loc[london["carriageway_hazards"].eq(0), "carriageway_hazards_label"] = "None"

# Fold sentinel codes (-1 "data missing or out of range", 99 "unknown") into one
# category so the model doesn't treat absent information as a real road condition.
label_predictors = [f"{col}_label" for col in decode_cols if col != "collision_severity"]
missing_like_labels = {
    "Data missing or out of range",
    "unknown (self reported)",
    "Unknown",
    "Unknown or other"
}

for col in label_predictors:
    london[col] = (
        london[col]
        .fillna("Missing/Unknown")
        .replace(list(missing_like_labels), "Missing/Unknown")
        .astype(str)
    )

### Selected variables

| Variable | Type | Role | Description | Notes |
|---|---|---|---|---|
| `severe` | Binary | Target | Fatal/serious versus slight collision | Created from `collision_severity` |
| `longitude`, `latitude` | Numeric | Spatial visualisation | Collision coordinates | Used for maps, not modelling |
| `borough_name` | Categorical | Spatial predictor | London borough / local authority | One-hot encoded in models |
| `speed_limit` | Numeric | Predictor | Posted speed limit | Standardised in models |
| `road_type_label` | Categorical | Predictor | Road type | Decoded and one-hot encoded |
| `junction_detail_label`, `junction_control_label` | Categorical | Predictor | Junction layout and control | Decoded and one-hot encoded |
| `pedestrian_crossing_label` | Categorical | Predictor | Pedestrian crossing context | Decoded and one-hot encoded |
| `light_conditions_label`, `weather_conditions_label`, `road_surface_conditions_label` | Categorical | Predictor | Environmental conditions | Decoded and one-hot encoded |
| `special_conditions_at_site_label`, `carriageway_hazards_label` | Categorical | Predictor | Site-specific hazards or conditions | Missing/unknown values retained as a category |
| `day_of_week`, `month`, `hour` | Categorical | Temporal predictors | Collision timing | One-hot encoded |
| `number_of_vehicles` | Numeric | Predictor | Vehicles involved in collision | Collision-context variable |

In [ ]:
selected_cols = [
    "severe", "longitude", "latitude", "speed_limit",
    "road_type_label", "junction_detail_label", "junction_control_label",
    "pedestrian_crossing_label", "light_conditions_label",
    "weather_conditions_label", "road_surface_conditions_label",
    "urban_or_rural_area_label", "trunk_road_flag_label",
    "day_of_week", "month", "hour", "number_of_vehicles"
]

data_audit = pd.DataFrame({
    "missing_count": london[selected_cols].isna().sum(),
    "missing_share": london[selected_cols].isna().mean().round(3),
    "unique_values": london[selected_cols].nunique()
})

display(data_audit)

numeric_summary = london[["speed_limit", "number_of_vehicles", "hour", "month"]].describe().round(2)
display(numeric_summary)

## Methodology

The workflow combines exploratory spatial description with interpretable classification. Temporal, road-type and borough patterns are first summarised using severe-collision *shares*, not raw counts, because categories differ in total collision volume and counts would mostly reflect exposure.

A balanced logistic regression then provides an interpretable baseline, and a class-weighted random forest is added to capture non-linear relationships between road, environmental, temporal and spatial predictors. Random forest hyperparameters are tuned with five-fold stratified cross-validation on the training data only, keeping the test set unseen during tuning. Random forest impurity-based feature importance is then used to summarise which predictors contribute most to the fitted classifier.

Train and test sets use stratification to preserve the severe / non-severe balance. Performance is reported with accuracy, precision, recall, F1-score and ROC-AUC. F1 is used as the primary tuning criterion because accuracy can be high even when severe collisions are mostly missed.

## Results and discussion

### Class balance

In [ ]:
# Class balance figure
severity_plot = pd.DataFrame({
    "severity_group": ["Slight", "Fatal/Serious"],
    "count": [
        (london["severe"] == 0).sum(),
        (london["severe"] == 1).sum()
    ]
})

severity_plot["share"] = severity_plot["count"] / severity_plot["count"].sum()

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(severity_plot["severity_group"], severity_plot["count"])

ax.set_title("Collision severity groups in London, 2024")
ax.set_xlabel("")
ax.set_ylabel("Number of collisions")

for i, row in severity_plot.iterrows():
    ax.text(
        i,
        row["count"],
        f"{row['share']:.1%}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

display(severity_plot)

Fatal and serious collisions make up 17.2% of the London subset. The imbalance is not extreme, but it is enough to make accuracy alone misleading.


### Temporal and road-environment patterns

In [ ]:
hourly_severity = (
    london
    .groupby("hour")
    .agg(
        total_collisions=("severe", "size"),
        severe_collisions=("severe", "sum"),
        severe_share=("severe", "mean")
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(hourly_severity["hour"], hourly_severity["severe_share"], marker="o")

ax.set_title("Share of severe collisions by hour of day")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Share of collisions that were fatal or serious")
ax.set_xticks(range(0, 24))
ax.yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

plt.tight_layout()
plt.show()

Severe-collision share is highest in late-night and early-morning hours and lower during much of the afternoon. Because the figure uses proportions, it describes the severity mix rather than collision volume by time.


In [ ]:
road_type_severity = (
    london
    .groupby("road_type_label")
    .agg(
        total_collisions=("severe", "size"),
        severe_collisions=("severe", "sum"),
        severe_share=("severe", "mean")
    )
    .reset_index()
    .sort_values("severe_share", ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(road_type_severity["road_type_label"], road_type_severity["severe_share"])

ax.set_title("Share of severe collisions by road type")
ax.set_xlabel("Share of collisions that were fatal or serious")
ax.set_ylabel("Road type")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

plt.tight_layout()
plt.show()

Road type separates severe-collision shares clearly. Dual carriageways and single carriageways have the highest severe shares, while roundabouts and slip roads are lower, plausibly reflecting lower speeds at some conflict points.


### Spatial pattern

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

slight = london[london["severe"] == 0]
severe_points = london[london["severe"] == 1]

ax.scatter(slight["longitude"], slight["latitude"], s=3, alpha=0.12, label="Slight")
ax.scatter(severe_points["longitude"], severe_points["latitude"], s=6, alpha=0.55, label="Fatal/Serious")

ax.set_title("Spatial distribution of reported road collisions in London, 2024")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(markerscale=3)

plt.tight_layout()
plt.show()

The point map shows severe collisions spread across London rather than concentrated in one small area. It is exploratory, not a formal risk map, because the data lack traffic-flow or road-length denominators.


In [ ]:
borough_name_map = {
    "E09000001": "City of London",
    "E09000002": "Barking and Dagenham",
    "E09000003": "Barnet",
    "E09000004": "Bexley",
    "E09000005": "Brent",
    "E09000006": "Bromley",
    "E09000007": "Camden",
    "E09000008": "Croydon",
    "E09000009": "Ealing",
    "E09000010": "Enfield",
    "E09000011": "Greenwich",
    "E09000012": "Hackney",
    "E09000013": "Hammersmith and Fulham",
    "E09000014": "Haringey",
    "E09000015": "Harrow",
    "E09000016": "Havering",
    "E09000017": "Hillingdon",
    "E09000018": "Hounslow",
    "E09000019": "Islington",
    "E09000020": "Kensington and Chelsea",
    "E09000021": "Kingston upon Thames",
    "E09000022": "Lambeth",
    "E09000023": "Lewisham",
    "E09000024": "Merton",
    "E09000025": "Newham",
    "E09000026": "Redbridge",
    "E09000027": "Richmond upon Thames",
    "E09000028": "Southwark",
    "E09000029": "Sutton",
    "E09000030": "Tower Hamlets",
    "E09000031": "Waltham Forest",
    "E09000032": "Wandsworth",
    "E09000033": "Westminster"
}

# drop EHEATHROW (33 rows, airport zone, not a London borough)
london = london[london["local_authority_ons_district"].isin(borough_name_map)].copy()
london["borough_name"] = london["local_authority_ons_district"].map(borough_name_map)

borough_severity = (
    london
    .groupby("borough_name")
    .agg(
        total_collisions=("severe", "size"),
        severe_collisions=("severe", "sum"),
        severe_share=("severe", "mean")
    )
    .reset_index()
)

# small denominators are unstable; drop boroughs with fewer than 300 collisions
borough_severity = borough_severity[borough_severity["total_collisions"] >= 300]
borough_severity = borough_severity.sort_values("severe_share", ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))

plot_df = borough_severity.head(15).sort_values("severe_share", ascending=True)
ax.barh(plot_df["borough_name"], plot_df["severe_share"])

ax.set_title("Top boroughs by severe-collision share")
ax.set_xlabel("Share of collisions that were fatal or serious")
ax.set_ylabel("Borough")
ax.xaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

plt.tight_layout()
plt.show()

Borough results use severe-collision shares and exclude boroughs with fewer than 300 collisions to avoid unstable proportions. The variation suggests spatial context matters, but it is not a true risk rate because exposure data are unavailable.


### Classification models

In [ ]:
numeric_features = ["speed_limit", "number_of_vehicles"]

categorical_features = [
    "road_type_label",
    "junction_detail_label",
    "junction_control_label",
    "pedestrian_crossing_label",
    "light_conditions_label",
    "weather_conditions_label",
    "road_surface_conditions_label",
    "special_conditions_at_site_label",
    "carriageway_hazards_label",
    "urban_or_rural_area_label",
    "trunk_road_flag_label",
    "day_of_week",
    "month",
    "hour",
    "borough_name"
]

model_features = numeric_features + categorical_features

X = london[model_features].copy()
for col in categorical_features:
    X[col] = X[col].astype(str)

y = london["severe"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

In [ ]:
log_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

rf_base = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=50,
        class_weight="balanced",
        random_state=42,
        n_jobs=1
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_grid = {
    "model__max_depth": [8, None],
    "model__min_samples_leaf": [10, 20]
}

rf_search = GridSearchCV(
    estimator=rf_base,
    param_grid=rf_grid,
    scoring="f1",
    cv=cv,
    n_jobs=1
)
rf_search.fit(X_train, y_train)
print("Best RF parameters:", rf_search.best_params_)
print(f"Best mean CV F1: {rf_search.best_score_:.3f}")

models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Balanced logistic regression": log_model,
    "Class-weighted random forest": rf_search.best_estimator_
}


def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = y_pred

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "fitted_model": model
    }


results = []
fitted_models = {}

for name, model in models.items():
    result = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    fitted_models[name] = result.pop("fitted_model")
    results.append(result)

performance = pd.DataFrame(results)
display(performance.round(3))

The majority baseline shows why accuracy is a weak guide: predicting every collision as slight gives high accuracy but zero recall for severe collisions. The class-weighted models reduce accuracy but identify many more fatal or serious outcomes. The random forest is carried forward because, after cross-validated tuning, it has the highest recall.


In [ ]:
best_model_name = "Class-weighted random forest"
best_model = fitted_models[best_model_name]

print(f"Confusion matrix model: {best_model_name}")

y_pred = best_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Slight", "Fatal/Serious"],
    cmap="Blues",
    ax=ax
)

ax.set_title(f"Confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()

The confusion matrix is shown for the class-weighted random forest, the model used in the next section for feature interpretation. The model identifies a large proportion of severe collisions, but precision is low: many collisions flagged as severe are in fact slight. The practical implication is that the model is not suitable for predicting individual severe collisions with confidence. It is more usefully read as an exploratory screening model that flags the road, junction, pedestrian-crossing and spatial conditions associated with higher modelled severity, which is closer to how road safety interventions are normally planned — at network or area level rather than crash by crash.

In [ ]:
# PR curve - more informative than ROC when the positive class is rare
from sklearn.metrics import precision_recall_curve, average_precision_score

y_score = best_model.predict_proba(X_test)[:, 1]
precisions, recalls, _ = precision_recall_curve(y_test, y_score)
ap = average_precision_score(y_test, y_score)
baseline = y_test.mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls, precisions, label=f"Random forest (AP = {ap:.3f})")
ax.axhline(
    baseline,
    color="grey",
    linestyle="--",
    label=f"Severe-class baseline ({baseline:.1%})"
)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-recall curve, class-weighted random forest")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"Average precision: {ap:.3f}")
print(f"Severe-collision baseline rate in test set: {baseline:.3f}")

The precision-recall curve is more informative than ROC for this imbalanced task because it shows how precision deteriorates as recall rises. Average precision is above the 17.2% no-skill baseline, indicating real signal, but the curve also shows that any operational use would require an explicit threshold choice.


### Model interpretation

In [ ]:
# Aggregate one-hot importance values back to one number per original variable.
rf_model = fitted_models["Class-weighted random forest"]

encoded_feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()
raw_importances = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "encoded_feature": encoded_feature_names,
    "importance": raw_importances
})


def recover_original_feature(encoded_name):
    if encoded_name.startswith("num__"):
        return encoded_name.replace("num__", "")
    if encoded_name.startswith("cat__"):
        name = encoded_name.replace("cat__", "")
        for col in sorted(categorical_features, key=len, reverse=True):
            if name.startswith(col + "_"):
                return col
    return encoded_name


importance_df["feature"] = importance_df["encoded_feature"].apply(recover_original_feature)

feature_importance = (
    importance_df
    .groupby("feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

display(feature_importance.head(10))

fig, ax = plt.subplots(figsize=(9, 6))
plot_df = feature_importance.head(10).sort_values("importance", ascending=True)
ax.barh(plot_df["feature"], plot_df["importance"])

ax.set_title("Random forest feature importance")
ax.set_xlabel("Aggregated impurity-based importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

The fitted random forest relies most on pedestrian crossing context, road type and number of vehicles. Carriageway hazards, borough, junction detail and site-specific conditions form the next tier. This links the borough-share analysis to the model: spatial context remains predictive after road and temporal variables are included.

The coding of missing values matters. Before consolidating sentinel codes, junction control appeared highly important; however, around one fifth of London records had missing junction-control values, so much of that signal was missingness rather than substantive junction control. After consolidation, its importance drops. These impurity-based scores describe the fitted model only, and are sensitive to correlated predictors and one-hot encoding. They should not be read as causal effects.


## Conclusion

Severe road collisions form a minority but substantial share of reported injury collisions in London in 2024, and their proportion varies by hour, road type and borough. The modelling results show that severity can be partly distinguished using recorded road, environmental, temporal and spatial predictors. Pedestrian crossing context, road type, number of vehicles, carriageway hazards and borough are the most informative variables in the tuned random forest. Recall reaches a useful level for exploration, but precision remains low, showing that severe outcomes are noisy and only partly captured by STATS19 attributes.

These results are predictive associations, not causal evidence. Key limitations are the absence of exposure data, possible under-reporting, and omission of damage-only collisions. Handling official missing-value codes also matters: collapsing -1 and 99 into a separate category changes which features appear important. For Vision Zero, the workflow is best read as a way to prioritise closer investigation, not as a deployable risk model.


## References

Department for Transport (2025a) *Road safety open data*. Available at: https://www.gov.uk/government/statistical-data-sets/road-safety-open-data (Accessed: 20 April 2025).

Department for Transport (2025b) *Road Safety Data*. Available at: https://findtransportdata.dft.gov.uk/dataset/road-safety-data (Accessed: 20 April 2025).

Edwards, J.B. (1998) ‘The relationship between road accident severity and recorded weather’, *Journal of Safety Research*, 29(4), pp.249–262.

Mannering, F.L. and Bhat, C.R. (2014) ‘Analytic methods in accident research: methodological frontier and future directions’, *Analytic Methods in Accident Research*, 1, pp.1–22.

Savolainen, P.T., Mannering, F.L., Lord, D. and Quddus, M.A. (2011) ‘The statistical analysis of highway crash-injury severities: a review and assessment of methodological alternatives’, *Accident Analysis & Prevention*, 43(5), pp.1666–1676.

Transport for London (n.d.) *Vision Zero for London*. Available at: https://tfl.gov.uk/corporate/safety-and-security/road-safety/vision-zero-for-london (Accessed: 20 April 2025).